# Question 3: Schedule with Timezone Constraint

In [6]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

## Data and Timezone Mapping

Timezones use standard US offsets for Nov–Dec 2025: Eastern (−5), Central (−6), Mountain (−7), Pacific (−8).

In [7]:
nba_sc = pd.read_csv("games.csv")
nba_sc["Date"] = pd.to_datetime(nba_sc["Date"])

team_tz = {
    "Atlanta Hawks": -5,
    "Boston Celtics": -5,
    "Brooklyn Nets": -5,
    "Chicago Bulls": -6,
    "Cleveland Cavaliers": -5,
    "Dallas Mavericks": -6,
    "Denver Nuggets": -7,
    "Golden State Warriors": -8,
    "Houston Rockets": -6,
    "Los Angeles Lakers": -8,
    "Miami Heat": -5,
    "Milwaukee Bucks": -6,
    "New York Knicks": -5,
    "Philadelphia 76ers": -5,
    "Phoenix Suns": -7,
    "Toronto Raptors": -5,
}

## Sets and Parameters

In [8]:
T = sorted(set(nba_sc["Home"]).union(set(nba_sc["Visitor"])))
F = sorted(nba_sc["Date"].unique())
Z = sorted(set(team_tz[t] for t in T))

home_teams_on_date = {f: sorted(nba_sc.loc[nba_sc["Date"] == f, "Home"].tolist()) for f in F}
away_teams_on_date = {f: sorted(nba_sc.loc[nba_sc["Date"] == f, "Visitor"].tolist()) for f in F}

played_dates = {
    t: sorted(nba_sc.loc[(nba_sc["Home"] == t) | (nba_sc["Visitor"] == t), "Date"].unique())
    for t in T
}

req_pair = nba_sc.groupby(["Home", "Visitor"]).size().to_dict()

print(f"Teams: {len(T)}, Dates: {len(F)}, Timezones: {Z}")

Teams: 16, Dates: 16, Timezones: [-8, -7, -6, -5]


## Model

**Decision variables:**
- $x_{ijf} = 1$ if team $i$ hosts team $j$ on date $f$ (only defined for original home/away slots)
- $y_{ifz} = 1$ if team $i$'s game on date $f$ is played in timezone $z$

**Constraints:**
- (A) Each original home slot is assigned exactly one opponent
- (B) Each original away slot is assigned exactly one host
- (C) Directed pair counts are preserved from the original schedule (Q2 constraints e–h)
- Linking: $y$ matches the timezone of whichever arena team $i$ plays in
- (Q3) For every team, every 3 consecutive games forbid timezone triplets $(z_1, z_2, z_3)$ with $|z_1-z_2| + |z_2-z_3| \geq 4$

In [ ]:
model = gp.Model("Q3")
model.setParam("OutputFlag", 1)

# x variables (sparse: only original home/away slots)
x = {}
for f in F:
    for i in home_teams_on_date[f]:
        for j in away_teams_on_date[f]:
            if i != j:
                x[i, j, f] = model.addVar(vtype=GRB.BINARY, name=f"x_{i}_{j}_{f.strftime('%Y%m%d')}")

# (A) each home slot: exactly one opponent
for f in F:
    for i in home_teams_on_date[f]:
        model.addConstr(gp.quicksum(x[i, j, f] for j in away_teams_on_date[f] if i != j) == 1)

# (B) each away slot: exactly one host
for f in F:
    for j in away_teams_on_date[f]:
        model.addConstr(gp.quicksum(x[i, j, f] for i in home_teams_on_date[f] if i != j) == 1)

# (C) preserve directed pair frequencies
for i in T:
    for j in T:
        if i != j:
            rhs = req_pair.get((i, j), 0)
            model.addConstr(gp.quicksum(x[i, j, f] for f in F if (i, j, f) in x) == rhs)

# y variables
y = {}
for i in T:
    for f in played_dates[i]:
        for z in Z:
            y[i, f, z] = model.addVar(vtype=GRB.BINARY, name=f"y_{i}_{f.strftime('%Y%m%d')}_{z}")


for i in T:
    for f in played_dates[i]:
        for z in Z:
            expr = gp.LinExpr()
            if i in home_teams_on_date[f] and team_tz[i] == z:
                expr += gp.quicksum(x[i, j, f] for j in away_teams_on_date[f] if i != j and (i, j, f) in x)
            if i in away_teams_on_date[f]:
                expr += gp.quicksum(x[j, i, f] for j in home_teams_on_date[f] if j != i and (j, i, f) in x and team_tz[j] == z)
            model.addConstr(y[i, f, z] == expr)
        model.addConstr(gp.quicksum(y[i, f, z] for z in Z) == 1)

# Q3 constraint: forbid bad timezone triplets across 3 consecutive games
bad_triplets = [(z1, z2, z3) for z1 in Z for z2 in Z for z3 in Z
                if abs(z1 - z2) + abs(z2 - z3) >= 4]

for i in T:
    d = sorted(played_dates[i])
    for k in range(len(d) - 2):
        f1, f2, f3 = d[k], d[k+1], d[k+2]
        for (z1, z2, z3) in bad_triplets:
            model.addConstr(y[i, f1, z1] + y[i, f2, z2] + y[i, f3, z3] <= 2)

model.setObjective(0, GRB.MINIMIZE)
model.optimize()

print(f"\nStatus: {model.status}  (2=Optimal, 3=Infeasible)")
print(f"Vars: {model.NumVars}, Constraints: {model.NumConstrs}")

Set parameter OutputFlag to value 1
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.3.0 25D125)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 4912 rows, 2048 columns and 16576 nonzeros (Min)
Model fingerprint: 0x08cfcb34
Model has 0 linear objective coefficients
Variable types: 0 continuous, 2048 integer (2048 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+00]

Presolve removed 2102 rows and 1467 columns
Presolve time: 0.00s

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 0

Model is infeasible
Best objective -, best bound -, gap -

Status: 3  (2=Optimal, 3=Infeasible)
Vars: 2048, Constraints: 4912


## Infeasibility Diagnosis (IIS)

In [10]:
if model.status == GRB.INFEASIBLE:
    model.computeIIS()
    model.write("q3_infeasible.ilp")
    print("IIS constraints:")
    for c in model.getConstrs():
        if c.IISConstr:
            print(f"  {c.ConstrName}")
elif model.status == GRB.OPTIMAL:
    schedule = [{"Date": f, "Home": i, "Away": j}
                for (i, j, f), var in x.items() if var.X > 0.5]
    schedule_df = pd.DataFrame(schedule).sort_values(["Date", "Home"]).reset_index(drop=True)
    schedule_df.to_csv("schedule_q3.csv", index=False)
    print(f"Feasible schedule with {len(schedule_df)} games saved to schedule_q3.csv")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.3.0 25D125)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads


Computing Irreducible Inconsistent Subsystem (IIS)...

           Constraints          |            Bounds           |  Runtime
      Min       Max     Guess   |   Min       Max     Guess   |
--------------------------------------------------------------------------
        0      2160         -         0         0         0           0s
       33        33        33         0         0         0           0s

IIS computed: 33 constraints, 0 bounds
IIS runtime: 0.05 seconds (0.04 work units)
IIS constraints:
  R33
  R36
  R40
  R44
  R50
  R53
  R271
  R274
  R285
  R298
  R406
  R410
  R420
  R924
  R997
  R1004
  R1007
  R1077
  R1078
  R1079
  R1080
  R1084
  R1087
  R1088
  R1089
  R1090
  R1236
  R1244
  R1246
  R1564
  R3014
  R3205
  R3597


## Conclusion

The model is infeasible. The IIS isolates 33 constraints whose intersection can't be satisfied, including three forbidden triplet constraints on Denver Nuggets, Golden State Warriors, and Los Angeles Lakers in their Nov 11 – Nov 15 window. Each of these WestCoast teams is forced into a Pacific/Mountain → Eastern → Pacific/Mountain pattern whose timezone jumps sum to 6, which violate the smaller than 4 rule.

Because Q2 fixes each team's home dates, away dates, and directed matchup counts from the original draft, there is no permutation of opponents that eliminates this pattern. Hence no feasible schedule exists under the additional timezone constraint.